# Pokemon (Dark / Ghost) — exploratory analysis

Data file: **`pokemon_dark_ghost_gen1_4.csv`** in the same folder as this notebook (`week5/`).

**If code cells do not run:** choose a **Python kernel** for the notebook (kernel picker in the notebook toolbar). Run the **first code cell** before the others so `df` exists. You need **`pandas`** and **`matplotlib`** installed in that environment (`pip install pandas matplotlib`).

**If cells run but you see no output:** click the **thin bar or chevron directly under the cell** to expand the output area (outputs are sometimes collapsed). Try **Notebook: Clear All Outputs**, then **Restart Kernel**, then run the first cell again. The code below also uses `print` / `display` / `plt.show()` so results are easier for the editor to show.

In [ ]:
%matplotlib inline
import io
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

# Jupyter's working directory is often the repo root, not week5 — try several locations
_csv = "pokemon_dark_ghost_gen1_4.csv"
_candidates = [
    Path(_csv),
    Path("week5") / _csv,
    Path("../week5") / _csv,
    Path("../week4") / _csv,
]
path = next((p.resolve() for p in _candidates if p.exists()), None)
if path is None:
    raise FileNotFoundError(
        f"Could not find {_csv}. Tried: {[str(p) for p in _candidates]}"
    )

df = pd.read_csv(path)
print("Loaded:", path)
print("Shape (rows, columns):", df.shape)

## 1. What does the data look like?

Use `head()` for a quick peek at rows and columns, and `info()` for dtypes and non-null counts.

In [ ]:
display(df.head())

In [ ]:
buf = io.StringIO()
df.info(buf=buf)
print(buf.getvalue())

## 2. Distribution of `attack` and `special_attack`

Summary stats plus histograms give a rough picture of how offensive stats spread in this slice.

In [ ]:
display(df[["attack", "special_attack"]].describe())

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
df["attack"].hist(bins=15, alpha=0.6, label="attack", ax=ax)
df["special_attack"].hist(bins=15, alpha=0.6, label="special_attack", ax=ax)
ax.set_xlabel("Stat value")
ax.set_ylabel("Count")
ax.legend()
ax.set_title("Distribution of attack vs special_attack")
plt.tight_layout()
plt.show()

## 3. Meaningful subset: **Dark** and **Ghost** (both in `types`), plus `base_experience`

The `types` column uses `|` between two typings when present. We keep rows whose typings include **both** `dark` and `ghost` (e.g. `dark|ghost`, `ghost|dark`). Then we inspect that subset, including `base_experience`.

In [ ]:
both_dark_ghost = df[
    df["types"].str.contains("dark", case=False, na=False)
    & df["types"].str.contains("ghost", case=False, na=False)
]
display(
    both_dark_ghost[
        ["name", "types", "base_experience", "attack", "special_attack"]
    ]
)

In [ ]:
both_dark_ghost["base_experience"].describe()

## 4. Group by a category — **Poison** in typings vs not, average `special_attack`

`base_experience` is numeric in this file, so “base_experience=poison” is interpreted as a **type** question: does the species typings include **poison**? We group by that yes/no category and take the mean `special_attack`.

In [ ]:
_poison_summary = (
    df.assign(
        poison_in_types=lambda d: d["types"]
        .str.contains("poison", case=False, na=False)
        .map({True: "types include poison", False: "no poison type"})
    )
    .groupby("poison_in_types", observed=True)["special_attack"]
    .agg(["mean", "count"])
    .round(2)
)
display(_poison_summary)

## 5. Missing values — where are they?

Per-column counts of nulls; all columns complete if every count is 0.

In [ ]:
display(df.isnull().sum().sort_values(ascending=False))

In [ ]:
incomplete = df.isnull().sum()
miss = incomplete[incomplete > 0]
if miss.empty:
    print("No incomplete columns (no nulls in any column).")
else:
    display(miss)